In [1]:
import sys
sys.path.append("..")

In [2]:
import pandas as pd
from pathlib import Path
from src.preprocessing import build_ticket_text

In [3]:
RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")

df = pd.read_csv(RAW_DIR / "dataset-tickets-multi-lang-4-20k.csv")
df = df[df["language"] == "en"].copy()
print("Start:", len(df))

df["text"] = [build_ticket_text(s, b) for s, b in zip(df["subject"], df["body"])]

# drop tickets too short to carry signal
df = df[df["text"].str.split().str.len() >= 5]
# drop near-duplicates (13 found during EDA)
df = df.loc[~df["text"].str[:150].duplicated()]

df = df[["text", "queue", "priority", "type"]].reset_index(drop=True)
print("After cleaning:", len(df))
print("\nQueue counts:\n", df["queue"].value_counts())

Start: 11923
After cleaning: 11901

Queue counts:
 queue
Technical Support                  3408
Product Support                    2230
Customer Service                   1856
IT Support                         1389
Billing and Payments               1293
Returns and Exchanges               580
Service Outages and Maintenance     442
Sales and Pre-Sales                 330
Human Resources                     205
General Inquiry                     168
Name: count, dtype: int64


In [4]:
for t in df["text"].head(3):
    print("\n" + "-"*70)
    print(t[:300])


----------------------------------------------------------------------
customer support inquiry seeking information on digital strategies that can aid in brand growth and details on the available services looking forward to learning more to help our business grow thank you and i look forward to hearing from you soon

----------------------------------------------------------------------
data analytics for investment i am contacting you to request information on data analytics tools that can be utilized with the eclipse ide for enhancing investment optimization i am seeking suggestions for tools that can aid in making data driven decisions particularly i am interested in tools that 

----------------------------------------------------------------------
security dear customer support i am reaching out to inquire about the security protocols you have in place to protect medical data as a valued customer i want to ensure that my sensitive health information is handled with the utmost car

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["queue"],
)

X_train, X_test = train_df["text"], test_df["text"]
y_train_q, y_test_q = train_df["queue"], test_df["queue"]
y_train_p, y_test_p = train_df["priority"], test_df["priority"]

print("Train:", len(train_df), "| Test:", len(test_df))
print("\nSmallest test class:", y_test_q.value_counts().min())

Train: 9520 | Test: 2381

Smallest test class: 34


In [6]:
from sklearn.metrics import accuracy_score, f1_score

maj_q, maj_p = y_train_q.mode()[0], y_train_p.mode()[0]

print(f"Queue    | majority '{maj_q}': "
      f"acc={accuracy_score(y_test_q, [maj_q]*len(y_test_q)):.3f}")
print(f"Priority | majority '{maj_p}': "
      f"acc={accuracy_score(y_test_p, [maj_p]*len(y_test_p)):.3f}")

Queue    | majority 'Technical Support': acc=0.286
Priority | majority 'medium': acc=0.419


In [7]:
queue_to_prio = train_df.groupby("queue")["priority"].agg(lambda s: s.mode()[0])
pred_p = y_test_q.map(queue_to_prio)

print(f"\nPriority from queue alone: "
      f"acc={accuracy_score(y_test_p, pred_p):.3f}, "
      f"macro-F1={f1_score(y_test_p, pred_p, average='macro'):.3f}")


Priority from queue alone: acc=0.531, macro-F1=0.433


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True,
    stop_words="english",
)

Xtr = vectorizer.fit_transform(X_train)   # fit on TRAIN only
Xte = vectorizer.transform(X_test)        # transform only — never fit

print("Train matrix:", Xtr.shape)
print("Test matrix :", Xte.shape)
print("Vocabulary  :", len(vectorizer.vocabulary_))
print("Sparsity    :", f"{100*(1 - Xtr.nnz/(Xtr.shape[0]*Xtr.shape[1])):.2f}%")

Train matrix: (9520, 20000)
Test matrix : (2381, 20000)
Vocabulary  : 20000
Sparsity    : 99.73%


In [9]:
import numpy as np
feats = np.array(vectorizer.get_feature_names_out())
print("Sample bigrams:", [f for f in feats if " " in f][:15])

Sample bigrams: ['ability deliver', 'ability effectively', 'ability make', 'ability manage', 'ability meet', 'ability optimize', 'ability provide', 'ability run', 'ability track', 'able assist', 'able identify', 'able launch', 'able offer', 'able pinpoint', 'able provide']


In [10]:
import joblib
PROC_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROC_DIR / "tickets_clean_en.csv", index=False)

joblib.dump(
    {"Xtr": Xtr, "Xte": Xte,
     "y_train_q": y_train_q, "y_test_q": y_test_q,
     "y_train_p": y_train_p, "y_test_p": y_test_p},
    PROC_DIR / "splits.joblib",
)
joblib.dump(vectorizer, "../models/tfidf_vectorizer.joblib")
print("Saved.")

Saved.
